# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** One row = one content item scored at a monthly snapshot: `(content_hash_id, snapshot t)`. Snapshots are per client, never on one global calendar date, because history depth differs widely between clients (`dim_clients.gsc_data_start` runs from 2025-01-27 to 2026-06-02).

**Windows for each snapshot t**
- Feature window: t-90d to t.
- Recent-trend window (features): the last 30 days (t-30 to t) vs the 30 days before that (t-60 to t-30).
- Label window: t to t+30d. It is used only to compute the outcome, never as a feature.

**Snapshots and split**
- Snapshots fall on month-ends. The earliest possible is 2025-04-30 (it needs 90 days of client history).
- Train: t = 2025-04-30 to 2026-03-31. Validation (model choice): t = 2026-04-30. Sealed test: t = 2026-05-31 (label window = June 2026), used once at the end.
- Each split's label windows end on or before the next split's snapshot date.
- t = 2026-06-30 has no label window. It is used only to produce the final ranked list.

**Modeling universe at snapshot t** (every condition is known at t and none touches the label window):
1. Published and not deleted in `dim_content`.
2. `content_created_date` is at least 90 days before t.
3. The client has at least 90 days of history at t (from `gsc_data_start`).
4. At least 100 impressions in the 30 days before t.

Pages that fail condition 4 are not labeled. They are reported as uncovered (count and share of active pages per snapshot) and appear in the final queue as "insufficient signal - monitor". The floor is applied to the pre-snapshot window only: applying it to the label window would drop pages that collapsed to zero and hide the declines the model is meant to find. The 100-impression floor is a design choice; sensitivity at 50 and 200 is reported in Results.

**Data quality note.** `fact_content_daily_performance` contains exact duplicate rows on `(report_date, client_hash_id, content_hash_id)`: 6,390 groups / 12,780 rows (0.016% of 78.8M). After `SELECT DISTINCT` no duplicate group remains, so every duplicate was a full-row copy. Every query applies `SELECT DISTINCT` before aggregation.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Panel range and size (metadata only, cheap)
con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
""").show()

# Grain as delivered: report_date x client x content should be unique, but is not
con.sql(f"""
    SELECT COUNT(*) AS n_dup_groups, SUM(c) AS n_rows_involved
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {TABLES['fact_daily']}
        GROUP BY 1,2,3
        HAVING c > 1
    )
""").show()

# Grain after SELECT DISTINCT: 0 groups left means every duplicate was an exact full-row copy
# (slowest cell in the notebook: it scans the whole table)
con.sql(f"""
    SELECT COUNT(*) AS n_dup_groups_after_distinct
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM (SELECT DISTINCT * FROM {TABLES['fact_daily']})
        GROUP BY 1,2,3
        HAVING COUNT(*) > 1
    )
""").show()

# Per-client history depth: why snapshots are per client, not global
con.sql(f"SELECT MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest FROM {TABLES['dim_clients']}").show()

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│   min_d    │   max_d    │    n     │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │ 78835655 │
└────────────┴────────────┴──────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬─────────────────┐
│ n_dup_groups │ n_rows_involved │
│    int64     │     int128      │
├──────────────┼─────────────────┤
│         6390 │           12780 │
└──────────────┴─────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────────┐
│ n_dup_groups_after_distinct │
│            int64            │
├─────────────────────────────┤
│                           0 │
└─────────────────────────────┘

┌────────────┬────────────┐
│  earliest  │   latest   │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-01-27 │ 2026-06-02 │
└────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Fields not listed below are not used.

### fact_content_daily_performance

| Bucket | Fields | Rule |
|---|---|---|
| Keys / context | `report_date`, `client_hash_id`, `content_hash_id` | Identify the row and build windows. Never model inputs. |
| Availability flags | `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` | Filters, not features. `ga4_data_available` is three-valued (TRUE / FALSE / NULL), so always test `IS TRUE`. |
| Label source | `gsc_impressions` summed over the forward window (t to t+30d) | Defines the outcome classes. Never appears in features. |
| Features (pre-snapshot windows only) | `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | Aggregated over t-90d to t, plus the last-30d vs prev-30d trend. Derived: CTR (clicks / impressions), impression-weighted position (sum_position / impressions), days with impressions, volatility. |
| Optional features (GA4) | `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `scroll_events` | Used only where `ga4_data_available IS TRUE`. Otherwise treated as missing, never as zero. |
| Excluded | `gsc_avg_position`, `month`, `sessions_ai`, `ai_*` | `gsc_avg_position` cannot be averaged correctly across days (use sum_position / impressions). `month` duplicates `report_date`. AI-referral sessions are too sparse to carry signal here. |

### dim_clients

| Bucket | Fields | Rule |
|---|---|---|
| Key | `client_hash_id` | Join key and grouping unit for robustness splits. |
| Snapshot eligibility | `gsc_data_start`, `ga4_data_start` | Decide whether a client has at least 90 days of history at snapshot t. Never features. |
| Context / filters | `has_gsc_access`, `has_ga4_access`, `access_profile` | Filters and slicing only. |
| Excluded | `is_active`, `client_created_date`, `client_updated_date` | Export-time status. `is_active` can encode the future (churned clients), and `client_updated_date` is an export timestamp. |

### dim_content

| Bucket | Fields | Rule |
|---|---|---|
| Keys | `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id` | Identifiers only, never features. |
| Static features (export-time) | `keyword_char_count`, `keyword_token_count`, `url_char_count`, `char_count`, `word_count`, `content_type`, `main_intent`, `competition`, `competition_level`, `search_volume`, `cpc`, `backlinks`, `category_count` | Stored once, as of export, so they may differ from what was true at t. `backlinks` is the riskiest: links gained after t can follow growth, so it may carry the outcome. Results are reported with and without these fields. |
| Time-aware feature | `content_created_date` | Creations are spread across the panel (4,299 to 56,593 pages per month, Jan 2025 to Jun 2026), so it behaves like a real creation date. Feature = page age at t. The page must exist at t and be at least 90 days old. |
| Excluded | `content_updated_date` | 82.5% of the 453,074 pages updated since Jan 2025 show an update in May to Jul 2026 (223,764 in May alone), so the value at any earlier t is unknown for most pages. Consistent with a bulk touch near export; the cause cannot be verified from this table. |
| Excluded (not profiled) | `keyword_created_date`, `optimization_eligible_date`, `provider_used`, `model_used` | Not checked against the panel; left out until they are. |
| Flag only (not a feature) | `last_optimized_date` | No recorded date is earlier than April 2026 (612 in April), so there is no training support. Used to tag `optimized_in_label_window` (t < date <= t+30d) for evaluation slices (headline metrics on untagged pages, sensitivity run on all pages) and as a "recently refreshed" reason code in the final queue. The field keeps one date per page, so the flag can miss earlier refreshes. |
| Universe filter | `is_published`, `is_deleted` | Modeling universe = published and not deleted (411,540 pages). Of the other 108,066 pages, 170 had impressions in 2026-03 (39,073 in total), against 176,568 pages and 280.6M impressions for the kept pages, so the filter adds negligible survivorship bias. Verified for 2026-03 only. |

### fact_query_90d

| Bucket | Fields | Rule |
|---|---|---|
| Excluded from model features | all `*_90d`, `*_last30`, `*_prev30`, `rare_*`, `anonymized_*` | One fixed window (2026-04-02 to 2026-06-30) that overlaps every label window from April on. Its window ends exactly at the final snapshot (2026-06-30), so it can add query context to the final ranked list, but it does not exist for earlier snapshots. |
| Keys | `client_hash_id`, `content_hash_id`, `query_hash_id` | Many rows per content item. `content_total_impressions_90d` repeats per row and must never be summed. |

In [3]:
# Column lists for every table used in the field tables above
for name in ['fact_daily', 'dim_clients', 'dim_content', 'fact_query_90d']:
    print(f"\n=== {name} ===")
    con.sql(f"DESCRIBE SELECT * FROM {TABLES[name]}").show(max_rows=100)


=== fact_daily ===
┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES     │ NULL    │ NULL

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

| Claim | Result |
|---|---|
| Panel range | 2025-01-27 to 2026-06-30; 78,835,655 rows |
| Grain (report_date, client, content) | Not unique as delivered: 6,390 duplicate groups (12,780 rows, 0.016%). After `SELECT DISTINCT`: 0 groups, so all were exact full-row copies. |
| Client history | `gsc_data_start` ranges from 2025-01-27 to 2026-06-02 |
| `dim_content` grain | 519,606 rows = 519,606 distinct (client, content) keys |
| Status | 411,540 published / not deleted (79.2%); 101,559 deleted (19.5%); 6,507 unpublished / not deleted (1.3%) |
| Traffic on deleted / unpublished pages (2026-03) | 170 pages, 39,073 impressions, vs 176,568 pages and 280.6M impressions for published / not deleted |
| Export-time dates | Run to 2026-07-06. `last_optimized_date`: 45,396 pages, all after 2026-03-31 (Apr 612, May 17,099, Jun 22,721, Jul 4,964). `content_updated_date`: 82.5% of the pages updated since Jan 2025 fall in May to Jul 2026. |
| `content_created_date` (published / not deleted) | 0 NULL; 5,832 before 2025; earliest 2024-11-22 |
| `fact_query_90d` window | Single window, 2026-04-02 to 2026-06-30 |
| Join coverage (2026-03-15) | 319,758 content keys, 0 missing from `dim_content` |
| Volume (2026-03, published / not deleted, at least 1 impression) | 176,568 pages; 30-day impressions quartiles 20 / 174 / 1,040; 42.6% below 100 |
| Eligible pages per snapshot (published / not deleted, age at least 90 days, client history at least 90 days, before the volume floor) | 2,726 (2025-04-30), 8,412, 17,787, 20,147, 20,942, then 68,359 (2025-09-30) rising to 293,683 (2026-05-31); 321,149 at 2026-06-30 |

In [4]:
# 1. dim_content grain: one row per (client, content)?
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id)) AS n_distinct_keys
    FROM {TABLES['dim_content']}
""").show()

# 2. Export-time status counts
con.sql(f"""
    SELECT is_published, is_deleted, COUNT(*) AS n
    FROM {TABLES['dim_content']}
    GROUP BY 1,2 ORDER BY n DESC
""").show()

# 3. Do date fields run past the panel end (2026-06-30)?
con.sql(f"""
    SELECT MAX(content_created_date)  AS max_created,
           MAX(content_updated_date)  AS max_updated,
           MAX(last_optimized_date)   AS max_optimized,
           SUM(CASE WHEN last_optimized_date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS optimized_after_mar31,
           COUNT(last_optimized_date) AS n_with_optimized_date
    FROM {TABLES['dim_content']}
""").show()

# 4. fact_query_90d window
con.sql(f"SELECT DISTINCT window_start, window_end FROM {TABLES['fact_query_90d']}").show()

# 5. Join coverage on one day: do daily-fact content keys exist in dim_content?
con.sql(f"""
    SELECT COUNT(*) AS n_fact_keys,
           SUM(CASE WHEN d.content_hash_id IS NULL THEN 1 ELSE 0 END) AS n_missing_in_dim
    FROM (SELECT DISTINCT client_hash_id, content_hash_id
          FROM {TABLES['fact_daily']}
          WHERE report_date = DATE '2026-03-15') f
    LEFT JOIN {TABLES['dim_content']} d
      ON f.client_hash_id = d.client_hash_id AND f.content_hash_id = d.content_hash_id
""").show()

# 6. content_updated_date and content_created_date by month (all pages)
con.sql(f"""
    SELECT date_trunc('month', content_updated_date) AS upd_month, COUNT(*) AS n
    FROM {TABLES['dim_content']}
    WHERE content_updated_date >= DATE '2025-01-01'
    GROUP BY 1 ORDER BY 1
""").show(max_rows=30)

con.sql(f"""
    SELECT date_trunc('month', content_created_date) AS created_month, COUNT(*) AS n
    FROM {TABLES['dim_content']}
    WHERE content_created_date >= DATE '2025-01-01'
    GROUP BY 1 ORDER BY 1
""").show(max_rows=30)

# 7. Created-date completeness inside the modeling universe
con.sql(f"""
    SELECT COUNT(*) AS n_pages,
           SUM(CASE WHEN content_created_date IS NULL THEN 1 ELSE 0 END) AS null_created,
           SUM(CASE WHEN content_created_date < DATE '2025-01-01' THEN 1 ELSE 0 END) AS created_before_2025,
           MIN(content_created_date) AS min_created
    FROM {TABLES['dim_content']}
    WHERE is_published AND NOT is_deleted
""").show()

# 8. Eligible pages per snapshot (conditions 1-3, before the volume floor)
con.sql(f"""
    WITH snaps(t) AS (VALUES
        (DATE '2025-04-30'), (DATE '2025-05-31'), (DATE '2025-06-30'), (DATE '2025-07-31'),
        (DATE '2025-08-31'), (DATE '2025-09-30'), (DATE '2025-10-31'), (DATE '2025-11-30'),
        (DATE '2025-12-31'), (DATE '2026-01-31'), (DATE '2026-02-28'), (DATE '2026-03-31'),
        (DATE '2026-04-30'), (DATE '2026-05-31'), (DATE '2026-06-30'))
    SELECT s.t, COUNT(*) AS n_eligible_pre_floor
    FROM snaps s
    JOIN {TABLES['dim_content']} d
      ON d.is_published AND NOT d.is_deleted
     AND d.content_created_date <= s.t - 90
    JOIN {TABLES['dim_clients']} c
      ON c.client_hash_id = d.client_hash_id
     AND c.gsc_data_start <= s.t - 90
    GROUP BY s.t
    ORDER BY s.t
""").show(max_rows=20)

┌────────┬─────────────────┐
│ n_rows │ n_distinct_keys │
│ int64  │      int64      │
├────────┼─────────────────┤
│ 519606 │          519606 │
└────────┴─────────────────┘

┌──────────────┬────────────┬────────┐
│ is_published │ is_deleted │   n    │
│   boolean    │  boolean   │ int64  │
├──────────────┼────────────┼────────┤
│ true         │ false      │ 411540 │
│ false        │ true       │ 101559 │
│ false        │ false      │   6507 │
└──────────────┴────────────┴────────┘

┌─────────────┬─────────────┬───────────────┬───────────────────────┬───────────────────────┐
│ max_created │ max_updated │ max_optimized │ optimized_after_mar31 │ n_with_optimized_date │
│    date     │    date     │     date      │        int128         │         int64         │
├─────────────┼─────────────┼───────────────┼───────────────────────┼───────────────────────┤
│ 2026-07-06  │ 2026-07-06  │ 2026-07-06    │                 45396 │                 45396 │
└─────────────┴─────────────┴─────────────

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬────────────┐
│ window_start │ window_end │
│     date     │    date    │
├──────────────┼────────────┤
│ 2026-04-02   │ 2026-06-30 │
└──────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬──────────────────┐
│ n_fact_keys │ n_missing_in_dim │
│    int64    │      int128      │
├─────────────┼──────────────────┤
│      319758 │                0 │
└─────────────┴──────────────────┘

┌────────────┬────────┐
│ upd_month  │   n    │
│    date    │ int64  │
├────────────┼────────┤
│ 2025-01-01 │   1441 │
│ 2025-03-01 │   1060 │
│ 2025-06-01 │   1455 │
│ 2025-07-01 │   3154 │
│ 2025-08-01 │   8734 │
│ 2025-09-01 │   5476 │
│ 2025-10-01 │   9660 │
│ 2025-11-01 │   3103 │
│ 2025-12-01 │    225 │
│ 2026-01-01 │   1185 │
│ 2026-02-01 │  33867 │
│ 2026-03-01 │    975 │
│ 2026-04-01 │   9034 │
│ 2026-05-01 │ 223764 │
│ 2026-06-01 │  79636 │
│ 2026-07-01 │  70305 │
├────────────┴────────┤
│ 16 rows   2 columns │
└─────────────────────┘

┌───────────────┬───────┐
│ created_month │   n   │
│     date      │ int64 │
├───────────────┼───────┤
│ 2025-01-01    │  7660 │
│ 2025-02-01    │  8047 │
│ 2025-03-01    │ 13370 │
│ 2025-04-01    │ 23427 │
│ 2025-05-01    │ 19028 │
│ 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Export-time metadata.** `dim_content` and `dim_clients` are one snapshot taken after the panel (dates run to 2026-07-06), not history as of each t. Static fields (`search_volume`, `competition`, `backlinks`, word counts) may differ from what was true at t, and `backlinks` may carry information from after t. Results are reported with and without these fields.
- **Refresh history is mostly invisible.** `last_optimized_date` keeps one date per page, and every recorded date falls between April and July 2026. Earlier optimizations may have been overwritten, so the recent clustering cannot be separated from overwriting, and any flag built from it can miss refreshes. Refresh effects cannot be separated from organic movement, and no causal claim is made.
- **Duplicates.** Exact duplicate rows exist in the daily fact table (0.016%); all queries use `SELECT DISTINCT`.
- **Coverage.** Pages under 100 impressions in the 30 days before t are not labeled (about 43% of active pages in 2026-03). The engine scores only pages with enough signal to judge.
- **Thin early snapshots.** Before the volume floor, the first five snapshots have 2,726 to 20,942 eligible pages against 68,359 to 293,683 later, so training is dominated by later months. A sensitivity run without the first five snapshots is planned.
- **Non-independent rows.** The same page appears in many snapshots. The main split is by time; a client-grouped holdout is reported as a robustness check.
- **Single sealed test month.** The test is one snapshot (t = 2026-05-31, label window June 2026). Results are observed and directional.
- **Excluded sources.** `fact_query_90d` (one window overlapping the label months), `content_updated_date` (bulk-touch pattern), and AI-referral fields (sparse). GA4 is optional and used only where `ga4_data_available IS TRUE`.
- **Created-date meaning.** `content_created_date` is the platform's creation date and may precede a page's first search visibility.
- **Observational data.** Findings are observed, measured associations used for decision support. Nothing here shows how Google ranks pages or what any refresh caused.

In [5]:
# A. When do recorded optimization dates fall?
con.sql(f"""
    SELECT date_trunc('month', last_optimized_date) AS opt_month, COUNT(*) AS n
    FROM {TABLES['dim_content']}
    WHERE last_optimized_date IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").show()

# B. Do deleted / unpublished pages carry traffic? (one mid-panel month)
con.sql(f"""
    SELECT d.is_published, d.is_deleted,
           COUNT(DISTINCT f.content_hash_id) AS n_content_with_impr,
           SUM(f.gsc_impressions) AS total_impr
    FROM (SELECT DISTINCT * FROM {TABLES['fact_daily']}
          WHERE month = '2026-03' AND gsc_impressions > 0) f
    JOIN {TABLES['dim_content']} d USING (client_hash_id, content_hash_id)
    GROUP BY 1,2 ORDER BY 1 DESC, 2
""").show()

# D. Volume distribution: 30-day impressions per published page (2026-03), to size the uncovered share
con.sql(f"""
    SELECT COUNT(*) AS n_pages,
           quantile_cont(imp, [0.25, 0.5, 0.75, 0.9]) AS q25_50_75_90,
           SUM(CASE WHEN imp >= 10  THEN 1 ELSE 0 END) AS n_ge_10,
           SUM(CASE WHEN imp >= 50  THEN 1 ELSE 0 END) AS n_ge_50,
           SUM(CASE WHEN imp >= 100 THEN 1 ELSE 0 END) AS n_ge_100
    FROM (
        SELECT f.client_hash_id, f.content_hash_id, SUM(f.gsc_impressions) AS imp
        FROM (SELECT DISTINCT * FROM {TABLES['fact_daily']} WHERE month = '2026-03') f
        JOIN {TABLES['dim_content']} d USING (client_hash_id, content_hash_id)
        WHERE d.is_published AND NOT d.is_deleted
        GROUP BY 1,2
        HAVING SUM(f.gsc_impressions) > 0
    )
""").show()

┌────────────┬───────┐
│ opt_month  │   n   │
│    date    │ int64 │
├────────────┼───────┤
│ 2026-04-01 │   612 │
│ 2026-05-01 │ 17099 │
│ 2026-06-01 │ 22721 │
│ 2026-07-01 │  4964 │
└────────────┴───────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬────────────┬─────────────────────┬────────────┐
│ is_published │ is_deleted │ n_content_with_impr │ total_impr │
│   boolean    │  boolean   │        int64        │   int128   │
├──────────────┼────────────┼─────────────────────┼────────────┤
│ true         │ false      │              176568 │  280618516 │
│ false        │ false      │                   8 │       7824 │
│ false        │ true       │                 162 │      31249 │
└──────────────┴────────────┴─────────────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────────────────────┬─────────┬─────────┬──────────┐
│ n_pages │         q25_50_75_90          │ n_ge_10 │ n_ge_50 │ n_ge_100 │
│  int64  │           double[]            │ int128  │ int128  │  int128  │
├─────────┼───────────────────────────────┼─────────┼─────────┼──────────┤
│  176568 │ [20.0, 174.0, 1040.0, 3934.0] │  143103 │  116063 │   101409 │
└─────────┴───────────────────────────────┴─────────┴─────────┴──────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.